# Cell-type composition, ratios and densities across segmentation methods

## 1. Imports, paths and analysis configuration

In [1]:
from pathlib import Path
import gzip, shutil, tempfile, gc, warnings
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from matplotlib.lines import Line2D
from shapely.ops import unary_union
from matplotlib.ticker import ScalarFormatter
from matplotlib.ticker import MaxNLocator


sns.set_theme(style="whitegrid", context="notebook")
BASE = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark/analysis/aging")
POLYGON_PATH = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark/misc/brain_regions/aging_brain_regions.parquet")
OUT = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark/misc/celltype_composition_out")
OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
BLUE_BRAIN_DENSITY_PATH = Path("/dss/dsshome1/05/ra57nur2/cellseg_benchmark/blue_brain_atlas/blue_brain_atlas_density.csv")
BLUE_BRAIN_COUNTS_PATH = Path("/dss/dsshome1/05/ra57nur2/cellseg_benchmark/blue_brain_atlas/blue_brain_atlas_counts.csv")
#SLIDE_THICKNESS_MM = 0.010
#CELL_DIAMETER_MM = {"Neurons": 0.010, "Glia": 0.008}
BASELINE_AGE = 3
AGE_ORDER = [3, 6, 12, 18, 24]


EXCLUDE_METHODS = set() #{"SIS_DAPI_total_mrna", "Watershed_Merlin"}
# Set to a list of names for a quick test; None processes all available methods.
METHODS_TO_RUN = None

## 2. Anatomical regions and cell-type definitions

`REMOVED_CELLTYPES` excludes Undefined, OECs and Low-Read-Cells at loading. Vascular is an overlapping summary of ECs, SMCs and Pericytes.

In [2]:
MERGED_AREAS = {"BS": ["BS"], "Cerebrum": ["CAsp", "CTX_1", "CTX_2", "DG-sg", "HIP", "STR_PAL", "VL", "V3"], "fiber_tracts": ["FT"]}
REMOVED_AREAS = {"Meninges", "MEN", "VS", "VS/STR", "BS/STR", "STR/CTX", "CTX/HIP"}
REMOVED_CELLTYPES = {"Undefined", "OECs", "Low-Read-Cells"}
GROUPED_CELL_TYPES = {
    "Neurons": ["Neurons-Gaba", "Neurons-Glut", "Neurons-Glyc-Gaba", "Neurons-Dopa", "Neurons-Granule-Immature", "Neurons-Other"],
    "Glia": ["Astrocytes", "Oligodendrocytes", "OPCs", "Microglia"],
    "Vascular": ["Pericytes", "ECs", "SMCs"],
    "Pericytes": ["Pericytes"], "ECs": ["ECs"], "SMCs": ["SMCs"]
}
AREA_LOOKUP = {fine: broad for broad, fines in MERGED_AREAS.items() for fine in fines}
CELL_LOOKUP = {fine: broad for broad, fines in GROUPED_CELL_TYPES.items() for fine in fines}
REGION_ORDER = list(MERGED_AREAS)
GROUP_ORDER = list(GROUPED_CELL_TYPES) + ["Other"]
print("Region lookup:", AREA_LOOKUP)
print("Cell groups:", GROUP_ORDER)
# Explicit vascular groups; all other retained cell types remain Other.
CELL_LOOKUP.update({"SMC": "SMCs", "Smooth Muscle Cells": "SMCs", "Endothelial cells": "ECs"})
VASCULAR_GROUPS = ["ECs", "SMCs", "Pericytes"]

DENSITY_GROUPS = ["Neurons", "Glia", "Vascular", "ECs", "SMCs", "Pericytes", "Total"]


Region lookup: {'BS': 'BS', 'CAsp': 'Cerebrum', 'CTX_1': 'Cerebrum', 'CTX_2': 'Cerebrum', 'DG-sg': 'Cerebrum', 'HIP': 'Cerebrum', 'STR_PAL': 'Cerebrum', 'VL': 'Cerebrum', 'V3': 'Cerebrum', 'FT': 'fiber_tracts'}
Cell groups: ['Neurons', 'Glia', 'Vascular', 'Pericytes', 'ECs', 'SMCs', 'Other']


## 3. Load and filter AnnData metadata

Retains original cell-ID matching, annotation QC and exclusion logic.

In [3]:
def read_adata_obs(path):
    """
    Read only the observation metadata from an AnnData HDF5 file.

    Supports HDF5 files even when they have a .h5ad.gz extension.
    """
    path = Path(path)
    adata = sc.read_h5ad(path, backed="r")

    try:
        obs = adata.obs.copy()
        obs.index.name = "cell_id"
        obs = obs.reset_index()
    finally:
        adata.file.close()

    return obs


def load_method(method_dir):
    """
    Load one segmentation method and match anatomical annotations
    to AnnData cell metadata.

    - Supports cell ID columns: cell_id, EntityID, Unnamed: 0
    - Supports sample columns: sample, slide
    - Validates cell IDs and sample assignments
    - Maps anatomical regions to BS, Cerebrum, fiber_tracts
    - Excludes removed regions and cell types
    - Assigns unlisted valid cell types to Other
    - Reports QC before filtering

    Requires:
        read_adata_obs()
        AREA_LOOKUP
        CELL_LOOKUP
        REMOVED_AREAS
        REMOVED_CELLTYPES
    """

    csv_path = method_dir / "brain_regions.csv"
    adata_path = method_dir / "adatas" / "adata_integrated.h5ad.gz"

    # --------------------------------------------------------
    # 1. Check required files
    # --------------------------------------------------------

    if not csv_path.exists() or not adata_path.exists():
        return None, {
            "method": method_dir.name,
            "status": "missing files",
            "missing_csv": not csv_path.exists(),
            "missing_adata": not adata_path.exists()
        }

    # --------------------------------------------------------
    # 2. Load anatomical annotations
    # --------------------------------------------------------

    annot = pd.read_csv(csv_path)

    id_columns = ["cell_id", "EntityID", "Unnamed: 0"]
    found = [col for col in id_columns if col in annot.columns]

    if len(found) != 1:
        raise ValueError(
            f"{method_dir.name}: Expected one cell ID column, "
            f"found {found}; available columns: {list(annot.columns)}"
        )

    annot = annot.rename(columns={found[0]: "cell_id"})

    if "sample" not in annot.columns and "slide" in annot.columns:
        annot = annot.rename(columns={"slide": "sample"})

    required = {"cell_id", "sample", "brain_region"}

    if not required.issubset(annot.columns):
        raise ValueError(
            f"{method_dir.name}: CSV missing "
            f"{required - set(annot.columns)}"
        )

    annot = annot[
        ["cell_id", "sample", "brain_region"]
    ].copy()

    # --------------------------------------------------------
    # 3. Load AnnData observation metadata
    # --------------------------------------------------------

    obs = read_adata_obs(adata_path)

    required_obs = {
        "cell_id",
        "sample",
        "cell_type_revised",
        "age_months"
    }

    if not required_obs.issubset(obs.columns):
        raise ValueError(
            f"{method_dir.name}: AnnData obs missing "
            f"{required_obs - set(obs.columns)}"
        )

    obs = obs[
        ["cell_id", "sample", "cell_type_revised", "age_months"]
    ].copy()

    # --------------------------------------------------------
    # 4. Standardize identifiers and check uniqueness
    # --------------------------------------------------------

    for frame in (obs, annot):
        frame["cell_id"] = frame["cell_id"].astype(str)
        frame["sample"] = frame["sample"].astype(str)

    for name, frame in [("AnnData", obs), ("CSV", annot)]:
        if frame.duplicated(["sample", "cell_id"]).any():
            raise ValueError(
                f"{method_dir.name}: Duplicate "
                f"(sample, cell_id) in {name}"
            )

    # --------------------------------------------------------
    # 5. Merge metadata and anatomical annotations
    # --------------------------------------------------------

    joined = obs.merge(
        annot,
        on=["sample", "cell_id"],
        how="left",
        validate="one_to_one",
        indicator=True
    )

    # --------------------------------------------------------
    # 6. QC before filtering
    # --------------------------------------------------------

    matched = int((joined["_merge"] == "both").sum())

    unmatched_ids = int(
        (joined["_merge"] == "left_only").sum()
    )

    extra_csv = len(annot) - matched

    missing_brain_region = int(
        joined["brain_region"].isna().sum()
    )

    missing_celltype = int(
        joined["cell_type_revised"].isna().sum()
    )

    # --------------------------------------------------------
    # 7. Map anatomical regions
    # --------------------------------------------------------

    joined["anatom_region"] = (
        joined["brain_region"].map(AREA_LOOKUP)
    )

    # Explicitly removed regions and missing annotations
    invalid_regions = (
        joined["brain_region"].isna()
        | joined["brain_region"].isin(REMOVED_AREAS)
    )

    # Regions not present in AREA_LOOKUP are also excluded.
    # Keep these separate in QC for detecting unexpected labels.
    unmapped_regions = (
        ~invalid_regions
        & joined["anatom_region"].isna()
    )

    # --------------------------------------------------------
    # 8. Map cell types
    # --------------------------------------------------------

    joined["cell_group"] = (
        joined["cell_type_revised"].map(CELL_LOOKUP)
    )

    # Exclude missing or explicitly removed cell types
    invalid_celltypes = (
        joined["cell_type_revised"].isna()
        | joined["cell_type_revised"].isin(REMOVED_CELLTYPES)
    )

    # All remaining unlisted cell types become Other
    joined.loc[
        ~invalid_celltypes & joined["cell_group"].isna(),
        "cell_group"
    ] = "Other"

    # --------------------------------------------------------
    # 9. Filter cells
    # --------------------------------------------------------

    valid_cells = (
        ~invalid_regions
        & ~unmapped_regions
        & ~invalid_celltypes
    )

    cells_filtered = joined.loc[valid_cells].copy()

    cells_filtered["method"] = method_dir.name

    cells_filtered = cells_filtered.drop(
        columns=["_merge"]
    )

    # --------------------------------------------------------
    # 10. Final QC
    # --------------------------------------------------------

    n_retained = len(cells_filtered)

    qc = {
        "method": method_dir.name,
        "status": "ok",

        "n_adata": len(obs),
        "n_csv": len(annot),

        "matched_cell_ids": matched,
        "unmatched_cell_ids": unmatched_ids,
        "extra_csv": extra_csv,

        "fraction_unmatched_ids":
            unmatched_ids / max(len(obs), 1),

        "missing_brain_region": missing_brain_region,
        "fraction_missing_region":
            missing_brain_region / max(len(obs), 1),

        "unmapped_regions": int(unmapped_regions.sum()),
        "removed_regions": int(
            joined["brain_region"].isin(REMOVED_AREAS).sum()
        ),

        "missing_celltype": missing_celltype,
        "removed_celltypes": int(
            joined["cell_type_revised"]
            .isin(REMOVED_CELLTYPES).sum()
        ),

        "n_retained": n_retained,
        "n_excluded": len(obs) - n_retained,
        "fraction_retained":
            n_retained / max(len(obs), 1)
    }

    return cells_filtered, qc


In [ ]:

# ------------------------------------------------------------
# 1. Identify candidate segmentation methods
# ------------------------------------------------------------

method_dirs = sorted(
    p for p in BASE.iterdir()
    if p.is_dir() and p.name not in EXCLUDE_METHODS
)

if METHODS_TO_RUN is not None:
    method_dirs = [
        p for p in method_dirs
        if p.name in set(METHODS_TO_RUN)
    ]

if METHODS_TO_RUN is not None:
    requested = set(METHODS_TO_RUN)
    available = {p.name for p in BASE.iterdir() if p.is_dir()}
    missing = requested - available
    if missing:
        raise ValueError(f"Unknown segmentation methods: {sorted(missing)}")
    excluded = requested & EXCLUDE_METHODS
    if excluded:
        print(f"Requested methods excluded by EXCLUDE_METHODS: {sorted(excluded)}")
print(f"Found {len(method_dirs)} candidate methods")


# ------------------------------------------------------------
# 2. Load, filter and aggregate each method
# ------------------------------------------------------------

original_tables = []
qc_records = []

for k, method_dir in enumerate(method_dirs, start=1):

    print(f"[{k}/{len(method_dirs)}] {method_dir.name}")

    cells, qc = load_method(method_dir)
    qc_records.append(qc)

    # Skip methods with missing files
    if cells is None:
        print(f"  Skipped: {qc['status']}")
        continue

    # load_method() already:
    # - removes missing and excluded anatomical regions
    # - removes excluded cell types
    # - assigns anatom_region
    # - assigns cell_group, including Other

    # Ensure age is numeric
    cells["age_months"] = pd.to_numeric(
        cells["age_months"],
        errors="coerce"
    )

    # Exclude cells without valid age information
    cells = cells.loc[
        cells["age_months"].notna()
    ].copy()

    # Aggregate cell counts by original anatomical region
    original = (
        cells.groupby(
            [
                "sample",
                "method",
                "age_months",
                "brain_region",
                "anatom_region",
                "cell_group"
            ],
            observed=True,
            dropna=False
        )
        .size()
        .rename("cell_count")
        .reset_index()
    )

    original_tables.append(original)

    del cells, original
    gc.collect()


# ------------------------------------------------------------
# 3. Save QC results
# ------------------------------------------------------------

qc_table = pd.DataFrame(qc_records)

display(qc_table)

qc_table.to_csv(
    OUT / "annotation_qc.csv",
    index=False
)


# ------------------------------------------------------------
# 4. Identify methods included in downstream analysis
# ------------------------------------------------------------

valid_methods = qc_table.loc[
    qc_table["status"] == "ok",
    "method"
].tolist()

excluded_methods = qc_table.loc[
    qc_table["status"] != "ok",
    "method"
].tolist()

print(f"\nMethods included: {len(valid_methods)}")
print(f"Methods excluded: {len(excluded_methods)}")

print("\nExcluded methods:")
for method in excluded_methods:
    print(f"  {method}")


# ------------------------------------------------------------
# 5. Combine aggregated results
# ------------------------------------------------------------

if not original_tables:
    raise RuntimeError(
        "No methods were successfully loaded. "
        "Check paths and QC output."
    )

composition_original_regions = pd.concat(
    original_tables,
    ignore_index=True
)

# Additional safety check
composition_original_regions = (
    composition_original_regions.loc[
        composition_original_regions["method"].isin(valid_methods)
    ].copy()
)

# Save results
composition_original_regions.to_csv(
    OUT / "composition_original_regions.csv",
    index=False
)

print(
    f"\nFinal composition table: "
    f"{len(composition_original_regions):,} rows"
)

print(
    f"Methods represented: "
    f"{composition_original_regions['method'].nunique()}"
)



Found 45 candidate methods
[1/45] Baysor_2D_Cellpose_1_DAPI_PolyT_0.2
[2/45] Baysor_2D_Cellpose_1_DAPI_PolyT_0.8
[3/45] Baysor_2D_Cellpose_1_DAPI_Transcripts_0.2
[4/45] Baysor_2D_Cellpose_1_DAPI_Transcripts_0.8
[5/45] Baysor_2D_Cellpose_1_nuclei_model_1.0
[6/45] Baysor_2D_Cellpose_2_DAPI_PolyT_0.2
[7/45] Baysor_2D_Cellpose_2_DAPI_PolyT_0.8
[8/45] Baysor_2D_Cellpose_2_DAPI_Transcripts_0.2
[9/45] Baysor_2D_Cellpose_2_DAPI_Transcripts_0.8
[10/45] Baysor_2D_denovo
[11/45] Baysor_3D_denovo
[12/45] CellSAM_DAPI
[13/45] CellSAM_DAPI_PolyT
[14/45] CellposeSAM_DAPI
[15/45] CellposeSAM_DAPI_PolyT
[16/45] Cellpose_1_DAPI_PolyT
[17/45] Cellpose_1_DAPI_Transcripts
[18/45] Cellpose_1_Merlin
[19/45] Cellpose_1_nuclei_model
[20/45] Cellpose_2_DAPI_PolyT
[21/45] Cellpose_2_DAPI_Transcripts
[22/45] ComSeg_Cellpose_1_nuclei_model
[23/45] Ficture_segments
[24/45] Ficture_segments_dapi
[25/45] Negative_Control_Rastered_10


## 4. Aggregate counts, calculate proportions and construct wide ratio table

Totals are computed from non-overlapping retained cell groups before Vascular is appended.

In [ ]:
# ------------------------------------------------------------
# Aggregate cell counts across merged anatomical regions
# ------------------------------------------------------------

keys = ["sample", "method", "age_months", "anatom_region"]

merged_counts = (
    composition_original_regions
    .groupby(
        keys + ["cell_group"],
        as_index=False,
        observed=True
    )
    .agg(cell_count=("cell_count", "sum"))
)

# ------------------------------------------------------------
# Total detected cells (without overlapping groups)
# ------------------------------------------------------------

# Exclude Vascular if already present to avoid double counting
base_counts = merged_counts.loc[
    merged_counts["cell_group"].ne("Vascular")
].copy()

all_counts = (
    base_counts
    .groupby(keys, as_index=False, observed=True)
    .agg(total_cells=("cell_count", "sum"))
)

# ------------------------------------------------------------
# Calculate combined Vascular cell counts
# ------------------------------------------------------------

vascular_counts = (
    base_counts.loc[
        base_counts["cell_group"].isin(VASCULAR_GROUPS)
    ]
    .groupby(keys, as_index=False, observed=True)
    .agg(cell_count=("cell_count", "sum"))
)

vascular_counts["cell_group"] = "Vascular"

# Add Vascular as an additional cell group
merged_counts = pd.concat(
    [base_counts, vascular_counts],
    ignore_index=True
)

# ------------------------------------------------------------
# Cell-type composition and proportions
# ------------------------------------------------------------

composition_merged_regions = merged_counts.merge(
    all_counts,
    on=keys,
    how="left",
    validate="many_to_one"
)

composition_merged_regions["proportion_all"] = (
    composition_merged_regions["cell_count"]
    / composition_merged_regions["total_cells"].replace(0, np.nan)
)

composition_merged_regions.to_csv(
    OUT / "composition_merged_regions.csv",
    index=False
)

# ------------------------------------------------------------
# Wide table: one row per sample × method × region × age
# ------------------------------------------------------------

wide = (
    merged_counts
    .pivot_table(
        index=keys,
        columns="cell_group",
        values="cell_count",
        aggfunc="sum",
        fill_value=0,
        observed=True
    )
    .reset_index()
)

wide.columns.name = None

# Ensure all required cell groups exist
for group in GROUP_ORDER:
    if group not in wide.columns:
        wide[group] = 0

wide = wide.merge(
    all_counts,
    on=keys,
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Cell-type ratios
# ------------------------------------------------------------

RATIOS = {
    "glia_neuron_ratio": ("Glia", "Neurons"),
    "pericyte_ec_ratio": ("Pericytes", "ECs"),
    "neuron_ec_ratio": ("Neurons", "ECs"),
    "neuron_smc_ratio": ("Neurons", "SMCs"),
    "neuron_pericyte_ratio": ("Neurons", "Pericytes"),
    "neuron_vascular_ratio": ("Neurons", "Vascular")
}

for ratio_name, (numerator, denominator) in RATIOS.items():
    wide[ratio_name] = (
        wide[numerator]
        / wide[denominator].replace(0, np.nan)
    )

# ------------------------------------------------------------
# Cell-type proportions
# ------------------------------------------------------------

for group in GROUP_ORDER:
    wide[f"{group}_proportion"] = (
        wide[group]
        / wide["total_cells"].replace(0, np.nan)
    )

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

wide.to_csv(
    OUT / "composition_ratios.csv",
    index=False
)

display(composition_merged_regions.head())
display(wide.head())

In [ ]:
assert (
    wide["Vascular"]
    == wide[["ECs", "SMCs", "Pericytes"]].sum(axis=1)
).all()

print("Vascular counts calculated correctly.")

## 5. Blue Brain reference ratios and reference data

Retains original Blue Brain parsing and reference tables.

In [ ]:

# ============================================================
# 1. Blue Brain reference data
# ============================================================

BLUE_BRAIN_AREA_LOOKUP = {
    "brain_stem": "BS",
    "cerebrum": "Cerebrum",
    "fibre_tracts": "fiber_tracts"
}


def parse_blue_brain_density(value):
    """Convert Blue Brain density values to numeric."""

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value.lower() in {"", "nan"}:
        return np.nan

    parts = value.split(".")

    if len(parts) > 2:
        return float("".join(parts[:-1]) + "." + parts[-1])

    return float(value)


blue_brain_density = None
blue_brain_ratios = None


# ------------------------------------------------------------
# 2. Load Blue Brain density reference
# ------------------------------------------------------------

if BLUE_BRAIN_DENSITY_PATH is not None:

    bb = pd.read_csv(
        BLUE_BRAIN_DENSITY_PATH
    ).drop(
        columns=["Unnamed: 0"],
        errors="ignore"
    )

    # Standardize anatomical region labels
    bb["Area"] = (
        bb["Area"]
        .astype("string")
        .str.strip()
        .replace(BLUE_BRAIN_AREA_LOOKUP)
    )

    # Convert density values to numeric
    for col in ["Neurons", "Glia"]:
        if col in bb.columns:
            bb[col] = bb[col].map(parse_blue_brain_density)

    blue_brain_density = bb.melt(
        id_vars="Area",
        value_vars=[
            col for col in ["Neurons", "Glia"]
            if col in bb.columns
        ],
        var_name="cell_group",
        value_name="blue_brain_cells_mm3"
    )


# ------------------------------------------------------------
# 3. Load Blue Brain absolute cell counts
# ------------------------------------------------------------

if BLUE_BRAIN_COUNTS_PATH is not None:

    bbcounts = pd.read_csv(
        BLUE_BRAIN_COUNTS_PATH
    ).drop(
        columns=["Unnamed: 0"],
        errors="ignore"
    )

    # Standardize anatomical region labels
    bbcounts["Area"] = (
        bbcounts["Area"]
        .astype("string")
        .str.strip()
        .replace(BLUE_BRAIN_AREA_LOOKUP)
    )

    # Convert reference counts to numeric
    for col in ["Neurons", "Glia"]:
        bbcounts[col] = pd.to_numeric(
            bbcounts[col],
            errors="coerce"
        )

    # Calculate reference Glia:Neuron ratio
    bbcounts["blue_brain_glia_neuron"] = (
        bbcounts["Glia"] /
        bbcounts["Neurons"].replace(0, np.nan)
    )

    blue_brain_ratios = bbcounts[
        ["Area", "blue_brain_glia_neuron"]
    ].copy()

    # Check that each anatomical region has one reference
    if blue_brain_ratios["Area"].duplicated().any():
        raise ValueError(
            "Blue Brain counts contain duplicate anatomical "
            "regions. Check the reference table."
        )


# ============================================================
# 4. MERFISH Glia:Neuron ratio
# ============================================================

# Keep only neurons and glia
comparison = composition_merged_regions.loc[
    composition_merged_regions["cell_group"].isin(
        ["Neurons", "Glia"]
    )
].copy()


# ------------------------------------------------------------
# 5. Convert to wide format
# ------------------------------------------------------------

comparison = (
    comparison.pivot_table(
        index=[
            "sample",
            "method",
            "age_months",
            "anatom_region"
        ],
        columns="cell_group",
        values="cell_count",
        aggfunc="sum",
        fill_value=0,
        observed=True
    )
    .reset_index()
)

comparison.columns.name = None

# Ensure both cell groups exist
for group in ["Neurons", "Glia"]:
    if group not in comparison.columns:
        comparison[group] = 0


# ------------------------------------------------------------
# 6. Calculate MERFISH Glia:Neuron ratio
# ------------------------------------------------------------

comparison["glia_neuron_ratio"] = (
    comparison["Glia"] /
    comparison["Neurons"].replace(0, np.nan)
)


# ============================================================
# 7. Add Blue Brain reference ratio
# ============================================================

if blue_brain_ratios is not None:

    comparison = comparison.merge(
        blue_brain_ratios,
        left_on="anatom_region",
        right_on="Area",
        how="left",
        validate="many_to_one"
    )

    # Check unmatched anatomical regions
    unmatched_regions = set(
        comparison.loc[
            comparison["Area"].isna(),
            "anatom_region"
        ].dropna()
    )

    if unmatched_regions:
        print(
            "Warning: No Blue Brain reference for:",
            sorted(unmatched_regions)
        )

    print(
        "Missing Blue Brain reference ratios:",
        comparison["blue_brain_glia_neuron"].isna().sum()
    )


# ============================================================
# 8. Display results
# ============================================================

print("\nMERFISH Glia:Neuron comparison")
print("Rows:", len(comparison))
print("Methods:", comparison["method"].nunique())
print("Samples:", comparison["sample"].nunique())

display(comparison.head())

if blue_brain_ratios is not None:

    print("\nBlue Brain reference ratios:")

    display(
        blue_brain_ratios.sort_values("Area")
    )



## 6. Plotting functions

One definition each of `baseline_horizontal()` and `age_trajectories()`. The latter shows sample-level mean ± SD.

In [ ]:
'''
def baseline_horizontal(data, metric, ylabel, stem, reference=None):
    """
    Plot sample-level values for each segmentation method at baseline age.

    Each anatomical region is shown in a separate panel.
    Methods are sorted by their region-specific mean.

    Coloured dots: individual biological samples.
    Black diamonds: mean ± SD across biological samples.
    Dashed lines: region-specific reference values.

    Scientific notation and 45-degree x-tick rotation are applied
    only to absolute cell counts (metric == "cell_count").

    Parameters
    ----------
    data : pd.DataFrame
        Sample-level data containing method, age_months,
        anatom_region and the requested metric.
    metric : str
        Column to plot.
    ylabel : str
        Label for the x-axis and figure title.
    stem : str
        Output filename stem.
    reference : str, dict or None
        Optional reference values:
        - str: column containing region-specific references
        - dict: mapping {anatom_region: reference_value}
        - None: no reference line
    """

    # --------------------------------------------------------
    # 1. Validate data and select baseline age
    # --------------------------------------------------------

    if metric not in data.columns:
        print(f"Skipping {metric}: column not found")
        return

    data = data.copy()

    data["age_months"] = pd.to_numeric(
        data["age_months"], errors="coerce"
    )

    data[metric] = pd.to_numeric(
        data[metric], errors="coerce"
    )

    data = data.loc[
        data["age_months"].eq(BASELINE_AGE)
        & data["anatom_region"].isin(REGION_ORDER)
        & data[metric].notna()
        & np.isfinite(data[metric])
    ].copy()

    regions = [
        r for r in REGION_ORDER
        if r in data["anatom_region"].unique()
    ]

    if not regions:
        print(f"No baseline data for {metric}")
        return

    # --------------------------------------------------------
    # 2. Prepare reference values
    # --------------------------------------------------------

    reference_values = {}

    if isinstance(reference, str):

        if reference not in data.columns:
            raise KeyError(
                f"Reference column '{reference}' not found."
            )

        reference_values = (
            data.groupby("anatom_region", observed=True)[reference]
            .first()
            .dropna()
            .to_dict()
        )

    elif isinstance(reference, dict):

        reference_values = reference.copy()

    elif reference is not None:

        raise TypeError(
            "reference must be a column name, dictionary or None"
        )

    # Blue Brain Glia:Neuron reference is undefined for
    # fiber tracts. Do not replace it with zero.
    if metric == "glia_neuron_ratio":
        reference_values.pop("fiber_tracts", None)

    reference_label = (
        "Blue Brain reference"
        if metric == "glia_neuron_ratio"
        else "3-month method median"
    )

    # --------------------------------------------------------
    # 3. Create figure
    # --------------------------------------------------------

    n_methods = data["method"].nunique()

    fig, axes = plt.subplots(
        1,
        len(regions),
        figsize=(
            max(8, 5 * len(regions)),
            max(3.5, 0.45 * n_methods + 1.5)
        ),
        squeeze=False
    )

    # --------------------------------------------------------
    # 4. Plot each anatomical region
    # --------------------------------------------------------

    for ax, region in zip(axes[0], regions):

        sub = data.loc[
            data["anatom_region"].eq(region)
        ].copy()

        # Sort methods by region-specific mean
        order = (
            sub.groupby("method", observed=True)[metric]
            .mean()
            .sort_values()
            .index
            .tolist()
        )

        if not order:
            ax.set_title(f"{region} — no data")
            continue

        

        # ----------------------------------------------------
        # Individual biological samples
        # ----------------------------------------------------

        sns.stripplot(
            data=sub,
            x=metric,
            y="method",
            order=order,
            ax=ax,
            size=8,
            alpha=0.45,
            color="tab:red",
            jitter=0
        )

        # ----------------------------------------------------
        # Mean ± SD across biological samples
        # ----------------------------------------------------

        summary = (
            sub.groupby("method", observed=True)[metric]
            .agg(mean="mean", sd="std")
            .reindex(order)
        )

        # SD is undefined for a single sample.
        summary["sd"] = summary["sd"].fillna(0)

        ax.errorbar(
            x=summary["mean"].values,
            y=np.arange(len(order)),
            xerr=summary["sd"].values,
            fmt="D",
            color="black",
            ecolor="black",
            markersize=6,
            elinewidth=1.5,
            capsize=4,
            capthick=1.5,
            linestyle="none",
            zorder=10
        )

        # ----------------------------------------------------
        # Region-specific reference line
        # ----------------------------------------------------

        ref_value = reference_values.get(region)

        if (
            ref_value is not None
            and np.isfinite(ref_value)
        ):

            ax.axvline(
                ref_value,
                color="black",
                linestyle="--",
                linewidth=1.5
            )

        # ----------------------------------------------------
        # 5. X-axis limits including SD and reference
        # ----------------------------------------------------

        values = sub[metric].to_numpy(dtype=float)
        values = values[np.isfinite(values)]

        lower_sd = summary["mean"] - summary["sd"]
        upper_sd = summary["mean"] + summary["sd"]

        values = np.concatenate([
            values,
            lower_sd.dropna().to_numpy(),
            upper_sd.dropna().to_numpy()
        ])

        if (
            ref_value is not None
            and np.isfinite(ref_value)
        ):
            values = np.append(values, float(ref_value))

        values = values[np.isfinite(values)]

        if len(values) > 0:

            xmin = values.min()
            xmax = values.max()

            span = xmax - xmin

            if span == 0:
                span = max(abs(xmax) * 0.2, 0.1)

            padding = span * 0.10

            ax.set_xlim(
                xmin - padding,
                xmax + padding
            )

            ax.set(
                title=region,
                xlabel=ylabel,
                ylabel="Segmentation method"
            )
            
            # Extra spacing for rotated x-axis labels
            ax.xaxis.labelpad = (
                25 if metric in ["cell_count", "cells_mm2"] else 12
            )
            
            # More readable x-axis ticks
            ax.xaxis.set_major_locator(
                MaxNLocator(nbins=6)
            )
            
            ax.tick_params(
                axis="x",
                labelsize=10
            )
            
            plt.setp(
                ax.get_xticklabels(),
                rotation=45,
                ha="right",
                rotation_mode="anchor"
            )
            
            ax.grid(
                axis="x",
                linestyle=":",
                alpha=0.35
            )
        # ----------------------------------------------------
        # 6. Scientific notation for absolute cell counts
        # ----------------------------------------------------

        if metric == "cell_count":

            formatter = ScalarFormatter(useMathText=True)
            formatter.set_scientific(True)
            formatter.set_powerlimits((0, 0))

            ax.xaxis.set_major_formatter(formatter)

            ax.tick_params(
                axis="x",
                labelsize=10
            )

            plt.setp(
                ax.get_xticklabels(),
                rotation=45,
                ha="right",
                rotation_mode="anchor"
            )

            ax.xaxis.get_offset_text().set_fontsize(10)

        # Remove subplot legends
        legend = ax.get_legend()

        if legend is not None:
            legend.remove()

    # --------------------------------------------------------
    # 7. Shared legend
    # --------------------------------------------------------

    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="None",
            markerfacecolor="tab:red",
            markeredgecolor="none",
            alpha=0.45,
            markersize=8,
            label="Individual sample"
        ),
        Line2D(
            [0], [0],
            marker="D",
            linestyle="-",
            color="black",
            markersize=6,
            linewidth=1.5,
            label="Mean ± SD"
        )
    ]

    # Only include the reference legend if at least
    # one region has a valid reference value.
    has_reference = any(
        reference_values.get(region) is not None
        and np.isfinite(reference_values[region])
        for region in regions
    )

    if has_reference:

        legend_handles.append(
            Line2D(
                [0], [0],
                color="black",
                linestyle="--",
                linewidth=1.5,
                label=reference_label
            )
        )

    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.965),
        ncol=len(legend_handles),
        frameon=False,
        fontsize=12
    )

    # --------------------------------------------------------
    # 8. Finalize and save
    # --------------------------------------------------------

    fig.suptitle(
        f"{ylabel} — {BASELINE_AGE} months",
        fontsize=17,
        y=1.02
    )

    fig.tight_layout(
        rect=[0, 0, 1, 0.91]
    )

    FIG.mkdir(parents=True, exist_ok=True)

    fig.savefig(
        FIG / f"{stem}.png",
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        FIG / f"{stem}.pdf",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)
'''

In [ ]:


def baseline_horizontal(data, metric, ylabel, stem, reference=None):

    # ----------------------------------------------------
    # 1. Prepare data
    # ----------------------------------------------------

    data = data.copy()

    data["age_months"] = pd.to_numeric(
        data["age_months"], errors="coerce"
    )

    data[metric] = pd.to_numeric(
        data[metric], errors="coerce"
    )

    data = data.loc[
        data["age_months"].eq(BASELINE_AGE)
        & data["anatom_region"].isin(REGION_ORDER)
        & np.isfinite(data[metric])
    ].copy()

    regions = [
        region for region in REGION_ORDER
        if region in data["anatom_region"].unique()
    ]

    if not regions:
        print(f"No baseline data for {metric}")
        return

    # ----------------------------------------------------
    # 2. Reference values
    # ----------------------------------------------------

    if isinstance(reference, str):

        if reference not in data.columns:
            raise KeyError(
                f"Reference column '{reference}' not found"
            )

        reference_values = (
            data.groupby("anatom_region", observed=True)[reference]
            .first()
            .dropna()
            .to_dict()
        )

    elif isinstance(reference, dict):
        reference_values = reference.copy()

    else:
        reference_values = {}

    # No Blue Brain Glia:Neuron reference for fiber tracts
    if metric == "glia_neuron_ratio":
        reference_values.pop("fiber_tracts", None)

    # ----------------------------------------------------
    # 3. Figure layout
    # ----------------------------------------------------

    n_methods = data["method"].nunique()
    n_regions = len(regions)

    fig = plt.figure(
        figsize=(
            9 * n_regions,
            max(7, 0.48 * n_methods + 3)
        )
    )

    outer = fig.add_gridspec(
        1,
        n_regions,
        left=0.15,
        right=0.98,
        bottom=0.14,
        top=0.78,
        wspace=1.5
    )

    for i, region in enumerate(regions):

        sub = data.loc[
            data["anatom_region"].eq(region)
        ].copy()

        # Order methods by their baseline mean
        order = (
            sub.groupby("method", observed=True)[metric]
            .mean()
            .sort_values()
            .index.tolist()
        )

        summary = (
            sub.groupby("method", observed=True)[metric]
            .agg(mean="mean", sd="std")
            .reindex(order)
        )

        summary["sd"] = summary["sd"].fillna(0)

        # ------------------------------------------------
        # 4. Determine whether a broken axis is appropriate
        # ------------------------------------------------

        control_name = "Negative_Control_Rastered_5"

        control_mask = sub["method"].eq(control_name)

        biological = sub.loc[~control_mask]
        control = sub.loc[control_mask]

        broken = False

        if (
            metric == "cells_mm2"
            and not biological.empty
            and not control.empty
        ):

            bio_summary = summary.loc[
                summary.index != control_name
            ]

            ctrl_summary = summary.loc[
                summary.index == control_name
            ]

            bio_max = max(
                biological[metric].max(),
                (bio_summary["mean"] + bio_summary["sd"]).max()
            )

            ctrl_min = min(
                control[metric].min(),
                (ctrl_summary["mean"] - ctrl_summary["sd"]).min()
            )

            broken = (
                np.isfinite(bio_max)
                and np.isfinite(ctrl_min)
                and ctrl_min > bio_max * 1.5
            )

        # ------------------------------------------------
        # 5. Create axes
        # ------------------------------------------------

        if broken:

            inner = outer[i].subgridspec(
                1, 2,
                width_ratios=[4, 1],
                wspace=0.10
            )

            ax = fig.add_subplot(inner[0])

            ax_right = fig.add_subplot(
                inner[1],
                sharey=ax
            )

            plot_axes = [ax, ax_right]

        else:

            ax = fig.add_subplot(outer[i])
            ax_right = None

            plot_axes = [ax]

        # ------------------------------------------------
        # 6. Plot individual samples and mean ± SD
        # ------------------------------------------------

        for current_ax in plot_axes:

            for j, method in enumerate(order):

                values = sub.loc[
                    sub["method"].eq(method),
                    metric
                ].to_numpy(dtype=float)

                current_ax.scatter(
                    values,
                    np.full(len(values), j),
                    s=42,
                    color="tab:red",
                    alpha=0.45,
                    zorder=3
                )

            current_ax.errorbar(
                summary["mean"].to_numpy(),
                np.arange(len(order)),
                xerr=summary["sd"].to_numpy(),
                fmt="D",
                color="black",
                ecolor="black",
                markersize=6,
                elinewidth=1.5,
                capsize=4,
                linestyle="none",
                zorder=5
            )

            ref_value = reference_values.get(region)

            if (
                ref_value is not None
                and np.isfinite(ref_value)
            ):
                current_ax.axvline(
                    float(ref_value),
                    color="black",
                    linestyle="--",
                    linewidth=1.5,
                    zorder=2
                )

            current_ax.grid(
                axis="x",
                linestyle=":",
                alpha=0.35
            )

            current_ax.xaxis.set_major_locator(
                MaxNLocator(nbins=5)
            )

            current_ax.tick_params(
                axis="x",
                labelsize=10
            )

            plt.setp(
                current_ax.get_xticklabels(),
                rotation=45,
                ha="right",
                rotation_mode="anchor"
            )

        # ------------------------------------------------
        # 7. Set x-axis limits
        # ------------------------------------------------

        if broken:

            bio_min = min(
                biological[metric].min(),
                (bio_summary["mean"] - bio_summary["sd"]).min()
            )

            ctrl_max = max(
                control[metric].max(),
                (ctrl_summary["mean"] + ctrl_summary["sd"]).max()
            )

            bio_span = max(bio_max - bio_min, 1)
            ctrl_span = max(ctrl_max - ctrl_min, 1)

            ax.set_xlim(
                max(0, bio_min - bio_span * 0.10),
                bio_max + bio_span * 0.15
            )

            ax_right.set_xlim(
                max(0, ctrl_min - ctrl_span * 0.15),
                ctrl_max + ctrl_span * 0.15
            )

            # Hide touching spines
            ax.spines["right"].set_visible(False)
            ax_right.spines["left"].set_visible(False)

            ax_right.tick_params(
                axis="y",
                left=False,
                labelleft=False
            )

            # Diagonal break marks
            d = 0.012

            kwargs = dict(
                color="black",
                clip_on=False,
                linewidth=1.3
            )

            ax.plot(
                (1-d, 1+d),
                (-d, +d),
                transform=ax.transAxes,
                **kwargs
            )

            ax.plot(
                (1-d, 1+d),
                (1-d, 1+d),
                transform=ax.transAxes,
                **kwargs
            )

            ax_right.plot(
                (-d, +d),
                (-d, +d),
                transform=ax_right.transAxes,
                **kwargs
            )

            ax_right.plot(
                (-d, +d),
                (1-d, 1+d),
                transform=ax_right.transAxes,
                **kwargs
            )

        else:

            values = sub[metric].to_numpy(dtype=float)

            values = np.concatenate([
                values,
                (
                    summary["mean"] - summary["sd"]
                ).to_numpy(),
                (
                    summary["mean"] + summary["sd"]
                ).to_numpy()
            ])

            ref_value = reference_values.get(region)

            if (
                ref_value is not None
                and np.isfinite(ref_value)
            ):
                values = np.append(
                    values, float(ref_value)
                )

            values = values[np.isfinite(values)]

            xmin = values.min()
            xmax = values.max()

            span = max(
                xmax - xmin,
                abs(xmax) * 0.2,
                0.1
            )

            ax.set_xlim(
                max(0, xmin - span * 0.10),
                xmax + span * 0.10
            )

        # ------------------------------------------------
        # 8. Labels and formatting
        # ------------------------------------------------

        ax.set_yticks(
            np.arange(len(order))
        )

        ax.set_yticklabels(
            order,
            fontsize=9
        )

        ax.invert_yaxis()

        ax.set_title(
            region,
            fontsize=14,
            pad=12
        )

        ax.set_ylabel("")

        ax.set_xlabel(
            ylabel,
            labelpad=25,
            fontsize=12
        )

        if ax_right is not None:

            ax_right.set_xlabel("")

            ax_right.xaxis.set_major_locator(
                MaxNLocator(nbins=3)
            )

        # Scientific notation for absolute cell counts
        if metric == "cell_count":

            for current_ax in plot_axes:

                formatter = ScalarFormatter(
                    useMathText=True
                )

                formatter.set_scientific(True)
                formatter.set_powerlimits((0, 0))

                current_ax.xaxis.set_major_formatter(
                    formatter
                )

    # ----------------------------------------------------
    # 9. Shared figure labels and legend
    # ----------------------------------------------------

    fig.supylabel(
        "Segmentation method",
        fontsize=13,
        x=0.01
    )

    handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="none",
            color="tab:red",
            alpha=0.45,
            markersize=7,
            label="Individual sample"
        ),
        Line2D(
            [0], [0],
            marker="D",
            color="black",
            linewidth=1.5,
            markersize=6,
            label="Mean ± SD"
        )
    ]

    if reference_values:

        handles.append(
            Line2D(
                [0], [0],
                color="black",
                linestyle="--",
                linewidth=1.5,
                label="Reference"
            )
        )

    fig.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.88),
        ncol=len(handles),
        frameon=False,
        fontsize=12
    )

    fig.suptitle(
        f"{ylabel} — {BASELINE_AGE} months",
        fontsize=18,
        y=0.97
    )

    # ----------------------------------------------------
    # 10. Save
    # ----------------------------------------------------

    FIG.mkdir(
        parents=True,
        exist_ok=True
    )

    fig.savefig(
        FIG / f"{stem}.png",
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        FIG / f"{stem}.pdf",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


In [ ]:
ALL_METHODS = sorted(
    composition_merged_regions["method"].unique()
)

METHOD_PALETTE = dict(zip(
    ALL_METHODS,
    sns.color_palette("husl", len(ALL_METHODS))
))


def age_trajectories(data, metric, ylabel, stem):

    # ----------------------------------------------------
    # Prepare data
    # ----------------------------------------------------

    data = data.copy()
    data["age_months"] = pd.to_numeric(
        data["age_months"], errors="coerce"
    )
    data[metric] = pd.to_numeric(
        data[metric], errors="coerce"
    )

    data = data.loc[
        data["anatom_region"].isin(REGION_ORDER)
        & np.isfinite(data[metric])
    ].copy()

    regions = [
        r for r in REGION_ORDER
        if r in data["anatom_region"].unique()
    ]

    if not regions:
        print(f"No valid data for {metric}")
        return

    # Order methods by baseline mean
    baseline = data.loc[
        data["age_months"].eq(BASELINE_AGE)
    ]

    method_order = (
        baseline.groupby("method", observed=True)[metric]
        .mean()
        .sort_values()
        .index.tolist()
    )

    method_order += [
        m for m in sorted(data["method"].unique())
        if m not in method_order
    ]

    # ----------------------------------------------------
    # Create figure
    # ----------------------------------------------------

    fig, axes = plt.subplots(
        1, len(regions),
        figsize=(6 * len(regions), 5),
        squeeze=False,
        sharey=True
    )

    # More evenly distributed colours for many methods
    palette = {
        method: METHOD_PALETTE[method]
        for method in method_order
    }
    
    # Additional differentiation through line styles
    line_styles = ["-", "--", "-.", ":"]
    
    style_map = {
        method: line_styles[i % len(line_styles)]
        for i, method in enumerate(method_order)
    }
    
    dash_patterns = {
        method: pattern
        for method, pattern in zip(
            method_order,
            ["", (5, 2), (3, 2, 1, 2), (1, 2)] * (
                (len(method_order) + 3) // 4
            )
        )
    }

    for ax, region in zip(axes[0], regions):

        sub = data.loc[
            data["anatom_region"].eq(region)
        ]

        # Mean ± SD across biological samples
        sns.lineplot(
            data=sub,
            x="age_months",
            y=metric,
            hue="method",
            hue_order=method_order,
            palette=palette,
            style="method",
            style_order=method_order,
            dashes=dash_patterns,
            markers=False,
            estimator="mean",
            errorbar="sd",
            err_style="bars",
            err_kws={
                "capsize": 3,
                "elinewidth": 1.2
            },
            marker="o",
            markersize=5,
            linewidth=2,
            ax=ax
        )

        # Individual biological samples
        sns.stripplot(
            data=sub,
            x="age_months",
            y=metric,
            color="grey",
            alpha=0.25,
            size=3,
            native_scale=True,
            ax=ax
        )

        ax.set_xticks(AGE_ORDER)
        ax.set_xlim(2, 25)

        ax.set(
            title=region,
            xlabel="Age (months)",
            ylabel=ylabel
        )

        ax.grid(True, alpha=0.25)

        # Scientific notation for absolute cell counts
        if metric == "cell_count":
            formatter = ScalarFormatter(useMathText=True)
            formatter.set_scientific(True)
            formatter.set_powerlimits((0, 0))
            ax.yaxis.set_major_formatter(formatter)

        if ax.get_legend():
            ax.get_legend().remove()

    # ----------------------------------------------------
    # Shared legend
    # ----------------------------------------------------

    handles = [
        Line2D(
            [0], [0],
            color=palette[method],
            linestyle=style_map[method],
            marker="o",
            linewidth=2,
            label=method
        )
        for method in method_order
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.05),
        ncol=min(3, len(handles)),
        frameon=False,
        fontsize=9
    )

    fig.suptitle(
        f"{ylabel} across ages",
        fontsize=17,
        fontweight="bold",
        y=1.02
    )

    fig.tight_layout(rect=[0, 0.12, 1, 0.95])

    # ----------------------------------------------------
    # Save
    # ----------------------------------------------------

    FIG.mkdir(parents=True, exist_ok=True)

    fig.savefig(
        FIG / f"{stem}.png",
        dpi=300,
        bbox_inches="tight"
    )
    fig.savefig(
        FIG / f"{stem}.pdf",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


## 7. Baseline ratios and all-method age trajectories

Baseline: Blue Brain Glia:Neuron reference; other ratios: median of method-level means at 3 months. Age trajectories: all segmentation methods.

In [ ]:
# ------------------------------------------------------------
# Baseline (3 months): All cell-type ratios
# ------------------------------------------------------------

RATIO_LABELS = {
    "glia_neuron_ratio": "Glia : Neuron",
    "pericyte_ec_ratio": "Pericyte : EC",
    "neuron_ec_ratio": "Neuron : EC",
    "neuron_smc_ratio": "Neuron : SMC",
    "neuron_pericyte_ratio": "Neuron : Pericyte",
    "neuron_vascular_ratio": "Neuron : Vascular",
}

# Blue Brain reference for Glia:Neuron
blue_brain_ref = None

if blue_brain_ratios is not None:
    blue_brain_ref = (
        blue_brain_ratios
        .set_index("Area")["blue_brain_glia_neuron"]
        .to_dict()
    )

# Plot every available ratio
for metric, label in RATIO_LABELS.items():

    if metric not in wide.columns:
        print(f"Skipping {metric}: column not found")
        continue

    if metric == "glia_neuron_ratio":
        reference = blue_brain_ref

    else:
        # Median of method-level means at baseline,
        # calculated separately for each brain area
        baseline = wide.loc[
            wide["age_months"].eq(BASELINE_AGE)
        ].copy()

        method_means = (
            baseline.groupby(
                ["anatom_region", "method"],
                observed=True
            )[metric]
            .mean()
            .reset_index()
        )

        reference = (
            method_means
            .groupby("anatom_region", observed=True)[metric]
            .median()
            .to_dict()
        )

    baseline_horizontal(
        wide,
        metric,
        label,
        f"baseline_{BASELINE_AGE}m_{metric}",
        reference=reference
    )


In [ ]:
for ratio, label in RATIO_LABELS.items():

    if ratio not in wide.columns:
        print(f"Skipping {ratio}: column not found")
        continue

    age_trajectories(
        data=wide,
        metric=ratio,
        ylabel=label,
        stem=f"age_{ratio}"
    )

## 8. Closest-versus-furthest segmentation method ratio trajectories

Retains separate ranking against the baseline reference and regional mean ± sample SD age plots.

In [ ]:
for metric, label in RATIO_LABELS.items():

    if metric not in wide.columns:
        print(f"Skipping {metric}: column not found")
        continue

    # ----------------------------------------------------
    # 1. Calculate baseline mean per method and region
    # ----------------------------------------------------
    
    baseline = wide.loc[
        (wide["age_months"] == BASELINE_AGE)
        & wide["anatom_region"].isin(REGION_ORDER)
    ].copy()
    
    method_means = (
        baseline.groupby(
            ["method", "anatom_region"],
            observed=True
        )[metric]
        .mean()
        .reset_index(name="method_mean")
    )
    
    # ----------------------------------------------------
    # 2. Define reference values
    # ----------------------------------------------------
    
    if metric == "glia_neuron_ratio":
    
        # Blue Brain reference for each anatomical region
        reference_values = (
            blue_brain_ratios
            .set_index("Area")["blue_brain_glia_neuron"]
            .to_dict()
        )
    
        reference_name = "Blue Brain"
    
    else:
    
        # Median across segmentation methods at 3 months,
        # calculated separately for each anatomical region
        reference_values = (
            method_means
            .groupby("anatom_region", observed=True)["method_mean"]
            .median()
            .to_dict()
        )
    
        reference_name = "3-month median"
    
    # ----------------------------------------------------
    # 3. Calculate distance from reference
    # ----------------------------------------------------
    
    method_means["reference"] = (
        method_means["anatom_region"].map(reference_values)
    )
    
    # Exclude undefined reference values (e.g. fiber tracts
    # for the Blue Brain Glia:Neuron ratio)
    method_means = method_means.dropna(
        subset=["method_mean", "reference"]
    )
    
    method_means["distance"] = (
        method_means["method_mean"]
        - method_means["reference"]
    ).abs()
    
    # ----------------------------------------------------
    # 4. Rank segmentation methods
    # ----------------------------------------------------
    
    method_ranking = (
        method_means
        .groupby("method", observed=True)
        .agg(
            mean_distance=("distance", "mean"),
            n_regions=("distance", "count")
        )
        .reset_index()
        .dropna(subset=["mean_distance"])
        .sort_values(["mean_distance", "method"])
    )
    
    if len(method_ranking) < 2:
        print(f"Skipping {metric}: fewer than two valid methods")
        continue
    
    closest_method = method_ranking.iloc[0]["method"]
    furthest_method = method_ranking.iloc[-1]["method"]
    
    selected_methods = [closest_method, furthest_method]
    
    print(f"\n{label} — reference: {reference_name}")
    print(f"Closest method:  {closest_method}")
    print(f"Furthest method: {furthest_method}")
    # ----------------------------------------------------
    # 3. Summarise age-dependent ratios
    # ----------------------------------------------------

    summary = (
        wide.loc[
            wide["method"].isin(selected_methods)
            & wide["anatom_region"].isin(REGION_ORDER)
        ]
        .groupby(
            ["method", "anatom_region", "age_months"],
            as_index=False,
            observed=True
        )[metric]
        .agg(
            mean_ratio="mean",
            sd_ratio="std",
            n_samples="count"
        )
    )

    summary["age_months"] = pd.to_numeric(
        summary["age_months"]
    )

    # ----------------------------------------------------
    # 4. Plot regional aging trajectories with mean ± SD
    # ----------------------------------------------------
    
    g = sns.relplot(
        data=summary,
        x="age_months",
        y="mean_ratio",
        hue="anatom_region",
        hue_order=REGION_ORDER,
        col="method",
        col_order=selected_methods,
        kind="line",
        marker="o",
        linewidth=2,
        markersize=7,
        height=4.2,
        aspect=1.2,
        facet_kws={"sharey": True}
    )
    
    # Retrieve the colours used for anatomical regions
    palette = dict(zip(
        REGION_ORDER,
        sns.color_palette(n_colors=len(REGION_ORDER))
    ))
    
    # Add SD error bars for each method × region × age
    for ax, method in zip(g.axes.flat, selected_methods):
    
        method_data = summary.loc[
            summary["method"].eq(method)
        ]
    
        for region in REGION_ORDER:
    
            region_data = method_data.loc[
                method_data["anatom_region"].eq(region)
            ].sort_values("age_months")
    
            if region_data.empty:
                continue
    
            ax.errorbar(
                x=region_data["age_months"],
                y=region_data["mean_ratio"],
                yerr=region_data["sd_ratio"].fillna(0),
                fmt="none",
                ecolor=palette[region],
                elinewidth=1.3,
                capsize=3,
                capthick=1.3,
                alpha=0.8,
                zorder=2
            )
    
        ax.set_xticks(AGE_ORDER)
        ax.set_xlim(2, 25)
        ax.tick_params(axis="both", labelsize=11)
        ax.grid(True, alpha=0.3)
    
    g.set_axis_labels(
        "Age (months)",
        f"{label} ratio"
    )
    
    g.set_titles(
        col_template="{col_name}",
        size=11
    )
    
    g.figure.suptitle(
        f"Regional aging differences — {label} ratio",
        fontsize=20,
        fontweight="bold",
        y=1.06
    )
    
    g.figure.subplots_adjust(
        top=0.83,
        wspace=0.25
    )

    # ----------------------------------------------------
    # 5. Save
    # ----------------------------------------------------

    stem = f"age_regional_closest_furthest_median_{metric}"

    g.figure.savefig(
        FIG / f"{stem}.png",
        dpi=300,
        bbox_inches="tight"
    )

    g.figure.savefig(
        FIG / f"{stem}.pdf",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(g.figure)

## 9. Baseline and age-dependent cell counts and proportions

Both plot types share the same seven groups and two metrics.

In [ ]:
CELL_GROUPS_TO_PLOT = ["Neurons", "Glia", "Vascular", "ECs", "SMCs", "Pericytes"]
METRICS = {"cell_count": "Absolute cell count", "proportion_all": "Cell proportion"}

for group in CELL_GROUPS_TO_PLOT:
    sub = composition_merged_regions.loc[
        composition_merged_regions["cell_group"].eq(group)
    ].copy()
    if sub.empty:
        print(f"Skipping {group}: no data")
        continue
    for metric, label in METRICS.items():
        baseline_horizontal(
            data=sub, metric=metric,
            ylabel=f"{group}: {label}",
            stem=f"baseline_3m_{group.lower()}_{metric}"
        )
        age_trajectories(
            data=sub, metric=metric,
            ylabel=f"{group}: {label}",
            stem=f"age_{group.lower()}_{metric}"
        )

## 10. Polygon areas and sample-level cell densities

Preserves polygon area sum (µm² to mm²), method/sample/region merge, and sample-level density summaries.

In [ ]:
# ------------------------------------------------------------
# 1. Load and validate polygons
# ------------------------------------------------------------

polygons = gpd.read_parquet(POLYGON_PATH)

print("Polygon columns:", list(polygons.columns))
print("Polygon CRS:", polygons.crs)

# Confirmed polygon schema
sample_col = "sample"
region_col = "label"

required_cols = [sample_col, region_col, polygons.geometry.name]

missing_cols = [
    col for col in required_cols
    if col not in polygons.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required polygon columns: {missing_cols}"
    )

if polygons.geometry.isna().any():
    raise ValueError("Polygon table contains missing geometries.")

if polygons.crs is not None and polygons.crs.is_geographic:
    raise ValueError(
        "Polygon coordinates are geographic degrees, "
        "not micrometres; convert before calculating areas."
    )

# ------------------------------------------------------------
# 2. Map original anatomical labels to merged brain areas
# ------------------------------------------------------------

# Uses the existing MERGED_AREAS definition.

areas = polygons[
    [sample_col, region_col, polygons.geometry.name]
].copy()

areas = areas.rename(
    columns={
        sample_col: "sample",
        region_col: "polygon_region"
    }
)

areas["sample"] = areas["sample"].astype(str)

areas["anatom_region"] = (
    areas["polygon_region"].map(AREA_LOOKUP)
)

# Retain already merged anatomical labels
areas["anatom_region"] = (
    areas["anatom_region"].fillna(
        areas["polygon_region"].where(
            areas["polygon_region"].isin(REGION_ORDER)
        )
    )
)

# Report unmapped labels
unmapped = sorted(
    areas.loc[
        areas["anatom_region"].isna(),
        "polygon_region"
    ].dropna().unique().tolist()
)

print("Unmapped polygon labels:", unmapped)

# Retain only analysis regions
areas = areas.loc[
    areas["anatom_region"].isin(REGION_ORDER)
].copy()

if areas.empty:
    raise ValueError(
        "No polygon regions match REGION_ORDER. "
        "Check MERGED_AREAS and polygon labels."
    )

# ------------------------------------------------------------
# 3. Calculate polygon areas (mm²)
# ------------------------------------------------------------

# Assumes polygon coordinates are measured in micrometres.
# CRS=None does not establish coordinate units.

areas["area_mm2"] = areas.geometry.area / 1e6

if (
    ~np.isfinite(areas["area_mm2"])
).any() or (areas["area_mm2"] <= 0).any():
    raise ValueError(
        "Invalid or non-positive polygon area detected."
    )

# Assumes constituent polygons do not overlap.
area_table = (
    areas.groupby(
        ["sample", "anatom_region"],
        observed=True,
        as_index=False
    )["area_mm2"]
    .sum()
)

area_table.to_csv(
    OUT / "polygon_areas_mm2.csv",
    index=False
)

print("\nArea ranges by anatomical region (mm²):")

display(
    area_table.groupby(
        "anatom_region",
        observed=True
    )["area_mm2"]
    .agg(["count", "min", "median", "max"])
)

# ------------------------------------------------------------
# 4. Prepare cell counts
# ------------------------------------------------------------

DENSITY_GROUPS = [
    "Neurons",
    "Glia",
    "Vascular",
    "ECs",
    "SMCs",
    "Pericytes",
    "Total"
]

density = wide[
    keys + ["Neurons", "Glia", "Vascular", "ECs", "SMCs", "Pericytes", "total_cells"]
].copy()

density["Total"] = density["total_cells"]

# ------------------------------------------------------------
# 5. Merge counts with polygon areas
# ------------------------------------------------------------

density["sample"] = density["sample"].astype(str)

density = density.merge(
    area_table,
    on=["sample", "anatom_region"],
    how="left",
    validate="many_to_one"
)

missing_area = (
    density.loc[
        density["area_mm2"].isna(),
        ["sample", "anatom_region"]
    ]
    .drop_duplicates()
)

if not missing_area.empty:

    print(
        "Missing polygon areas for these "
        "sample-region combinations:"
    )

    display(missing_area)

    raise ValueError(
        "Missing polygon areas for sample-region "
        "combinations; resolve before plotting densities."
    )

# ------------------------------------------------------------
# 6. Calculate cell densities (cells/mm²)
# ------------------------------------------------------------

for group in DENSITY_GROUPS:

    density[f"{group}_cells_mm2"] = (
        density[group] / density["area_mm2"]
    )

# ------------------------------------------------------------
# 7. Convert to long format
# ------------------------------------------------------------

density_long = density.melt(
    id_vars=keys + ["area_mm2"],
    value_vars=[
        f"{group}_cells_mm2"
        for group in DENSITY_GROUPS
    ],
    var_name="cell_group",
    value_name="cells_mm2"
)

density_long["cell_group"] = (
    density_long["cell_group"]
    .str.removesuffix("_cells_mm2")
)

density_long.to_csv(
    OUT / "sample_cell_densities_mm2.csv",
    index=False
)

print("\nSample-level cell density data:")
display(density_long.head())

# ------------------------------------------------------------
# 8. Summary by method, age, region and cell group
# ------------------------------------------------------------

# Each observation represents one biological sample.
# Mean and SD are calculated across samples within
# each segmentation method, age, region and cell group.

density_summary = (
    density_long
    .groupby(
        [
            "method",
            "age_months",
            "anatom_region",
            "cell_group"
        ],
        observed=True
    )
    .agg(
        n_samples=("cells_mm2", "count"),
        mean_cells_mm2=("cells_mm2", "mean"),
        sd_cells_mm2=("cells_mm2", "std")
    )
    .reset_index()
)

density_summary.to_csv(
    OUT / "cell_density_summary_by_method_age_region.csv",
    index=False
)

print("\nDensity summary by method, age, region and cell group:")
display(density_summary.head(15))

# ------------------------------------------------------------
# 9. Baseline density summary (3 months)
# ------------------------------------------------------------

baseline_density_summary = (
    density_summary.loc[
        density_summary["age_months"].eq(BASELINE_AGE)
    ]
    .copy()
)

baseline_density_summary.to_csv(
    OUT / "cell_density_summary_3months.csv",
    index=False
)

print(f"\nBaseline density summary ({BASELINE_AGE} months):")
display(baseline_density_summary.head(15))

# ------------------------------------------------------------
# 10. Overall diagnostic summary
# ------------------------------------------------------------

# This pools ages and methods and is intended only
# as a diagnostic, not for biological interpretation.

print("\nOverall density ranges (diagnostic only):")

display(
    density_long.groupby(
        ["anatom_region", "cell_group"],
        observed=True
    )["cells_mm2"]
    .agg(["count", "mean", "std", "min", "max"])
)


## 11. Baseline and age-dependent cell densities

Plots all seven density groups using the same baseline and age trajectory functions.

In [ ]:
for group in DENSITY_GROUPS:

    sub = density_long.loc[
        density_long["cell_group"].eq(group)
    ].copy()

    if sub.empty:
        print(f"Skipping {group}: no density data")
        continue

    # Baseline density (3 months)
    baseline_horizontal(
        data=sub,
        metric="cells_mm2",
        ylabel=f"{group}: Cell density (cells/mm²)",
        stem=f"baseline_3m_{group.lower()}_density"
    )

    # Age-dependent density
    age_trajectories(
        data=sub,
        metric="cells_mm2",
        ylabel=f"{group}: Cell density (cells/mm²)",
        stem=f"age_{group.lower()}_density"
    )

## 12. Density and polygon QC

Overlap diagnostic; the original area calculation assumes non-overlapping polygons.

In [ ]:
print("Polygon coordinate bounds:")
print(polygons.total_bounds)

print("\nBounds for one sample:")
sample = polygons["sample"].iloc[0]

sample_polygons = polygons.loc[
    polygons["sample"].eq(sample)
]

print("Sample:", sample)
print(sample_polygons.total_bounds)

print("\nPolygon area statistics (raw coordinate units²):")
print(polygons.geometry.area.describe())

In [ ]:


overlap_results = []

for (sample, region), group in areas.groupby(
    ["sample", "anatom_region"],
    observed=True
):

    geometries = group.geometry.tolist()

    # Sum of individual polygon areas
    sum_area = group.geometry.area.sum()

    # Area covered by at least one polygon
    union_geometry = unary_union(geometries)
    union_area = union_geometry.area

    # Area counted more than once
    overlap_area = max(0, sum_area - union_area)

    overlap_pct = (
        100 * overlap_area / sum_area
        if sum_area > 0 else np.nan
    )

    overlap_results.append({
        "sample": sample,
        "anatom_region": region,
        "n_polygons": len(group),
        "sum_area_mm2": sum_area / 1e6,
        "union_area_mm2": union_area / 1e6,
        "overlap_mm2": overlap_area / 1e6,
        "overlap_pct": overlap_pct
    })

overlap_check = pd.DataFrame(overlap_results)

display(
    overlap_check.sort_values(
        "overlap_pct",
        ascending=False
    )
)

print("\nMaximum overlap (%):")
print(overlap_check["overlap_pct"].max())